# EuroSAT land-use classification: full pipeline

Runtime → Change runtime type → **T4 GPU**. Run the cells top to bottom. With MobileNetV2 at 224 px, each RGB run takes roughly 20–40 min on a T4.

Results are saved to Google Drive so a Colab disconnect doesn't lose them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -b eurosat-landuse https://github.com/aswin-k-2005/vanaushadi.git /content/vanaushadi
%cd /content/vanaushadi/eurosat
!pip -q install tifffile

import os
os.environ['EUROSAT_DATA'] = '/content/data'                       # fast local disk
os.environ['EUROSAT_OUTPUT'] = '/content/drive/MyDrive/eurosat_outputs'  # persistent

## 1. Download + fixed split
Creates `split.csv` once. Every later experiment reuses it.

In [ ]:
!python data.py --composite rgb

## 2. Baseline: RGB + MobileNetV2

In [ ]:
!python train.py --composite rgb --backbone mobilenetv2

In [ ]:
RUN = '/content/drive/MyDrive/eurosat_outputs/rgb_mobilenetv2'
!python evaluate.py --run $RUN
!python evaluate.py --run $RUN --tta
from IPython.display import Image, display
display(Image(f'{RUN}/confusion_matrix.png'))

## 3. Confusability study (top 3 confused pairs + the classic ones)

In [ ]:
!python confusability.py --run $RUN --top 3
!python confusability.py --run $RUN --pair AnnualCrop PermanentCrop
!python confusability.py --run $RUN --pair Highway River
import pandas as pd
pd.read_csv(f'{RUN}/confusability/summary.csv')

In [ ]:
display(Image(f'{RUN}/confusability/AnnualCrop__PermanentCrop/gradcam_grid.png'))
display(Image(f'{RUN}/confusability/AnnualCrop__PermanentCrop/input_statistics.png'))

## 4. Change detection: synthetic benchmark with known ground truth

In [ ]:
!python change_detection.py --run $RUN --synthetic --scenes 100
display(Image(f'{RUN}/change_detection/synthetic_benchmark.png'))
display(Image(f'{RUN}/change_detection/synthetic_example.png'))

## 5. Change detection: real Sentinel-2 imagery
Export two true-colour images of the same area from [Copernicus Browser](https://browser.dataspace.copernicus.eu) (Sentinel-2 L2A, low cloud cover, same season, ideally same month in different years). Export at 10 m resolution with the same bounding box, upload them, then run:

In [ ]:
# from google.colab import files; files.upload()
# !python change_detection.py --run $RUN --before before.png --after after.png --gsd 10

## 6. Spectral experiment (optional, ~2 GB download)
Does near-infrared fix the crop/vegetation confusions? All three runs use the same 13-band source, split, backbone and recipe. Only the 3 bands fed to the network change.

In [ ]:
for c in ['ms_rgb', 'ms_falsecolor', 'ms_swir']:
    !python train.py --composite {c}
    !python evaluate.py --run /content/drive/MyDrive/eurosat_outputs/{c}_mobilenetv2

!python compare.py /content/drive/MyDrive/eurosat_outputs/ms_rgb_mobilenetv2 /content/drive/MyDrive/eurosat_outputs/ms_falsecolor_mobilenetv2 /content/drive/MyDrive/eurosat_outputs/ms_swir_mobilenetv2

## 7. Optional ablations for the report

In [ ]:
# Backbone comparison
# !python train.py --backbone efficientnetb0
# Pretraining vs from scratch
# !python train.py --weights none --out /content/drive/MyDrive/eurosat_outputs/rgb_scratch
# Label efficiency: 10% of training data
# !python train.py --train-fraction 0.1 --out /content/drive/MyDrive/eurosat_outputs/rgb_frac10

## 8. Run the web app locally
Copy `rgb_mobilenetv2/` (with `best_model.keras` and `run_config.json`) to `eurosat/outputs/` on your laptop, then run `python app.py`.